# Imports e información del sistema

In [4]:
import polars as pl
import pandas as pd
import time
import psutil
import os
import matplotlib.pyplot as plt
import numpy as np

print(f"CPU Cores: {psutil.cpu_count(logical=False)}")
print(f"RAM Total: {psutil.virtual_memory().total / (1024**3):.2f} GB")
print(f"Dataset Size: {os.path.getsize('../taxi_filtrado.parquet') / (1024**2):.2f} MB")

CPU Cores: 4
RAM Total: 7.64 GB
Dataset Size: 227.78 MB


# Definición de pipelines idénticos

In [5]:
file_path = '../taxi_filtrado.parquet'

# Verificar nombres exactos de las columnas
df_check = pd.read_parquet(file_path)
print("Columnas disponibles:", df_check.columns.tolist())

def run_polars_pipeline():
    t0 = time.time()
    df = pl.read_parquet(file_path)
    t_read = time.time() - t0

    t0 = time.time()
    df = df.filter((pl.col('trip_distance') > 0) & (pl.col('total_amount') > 0))
    t_filter = time.time() - t0

    t0 = time.time()
    agg = df.group_by('passenger_count').agg(pl.col('total_amount').mean().alias('avg_total_amount'))
    t_agg = time.time() - t0

    t0 = time.time()
    df = df.join(agg, on='passenger_count', how='left')
    t_join = time.time() - t0

    t0 = time.time()
    df = df.with_columns((pl.col('total_amount') / pl.col('trip_distance')).alias('cost_per_mile'))
    t_feat = time.time() - t0

    return {'Read': t_read, 'Filter': t_filter, 'Agg': t_agg, 'Join': t_join, 'Feat': t_feat}

def run_pandas_pipeline():
    t0 = time.time()
    df = pd.read_parquet(file_path).copy()
    t_read = time.time() - t0

    t0 = time.time()
    df = df[(df['trip_distance'] > 0) & (df['total_amount'] > 0)]
    t_filter = time.time() - t0

    t0 = time.time()
    agg = df.groupby('passenger_count')['total_amount'].mean().reset_index(name='avg_total_amount')
    t_agg = time.time() - t0

    t0 = time.time()
    df = df.merge(agg, on='passenger_count', how='left')
    t_join = time.time() - t0

    t0 = time.time()
    df['cost_per_mile'] = df['total_amount'] / df['trip_distance']
    t_feat = time.time() - t0

    return {'Read': t_read, 'Filter': t_filter, 'Agg': t_agg, 'Join': t_join, 'Feat': t_feat}

Columnas disponibles: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude', 'RatecodeID', 'store_and_fwd_flag', 'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount']


# Ejecución y tabla de resultados

In [ ]:
print("Ejecutando Polars...")
polars_times = run_polars_pipeline()

print("Ejecutando Pandas...")
pandas_times = run_pandas_pipeline()

df_results = pl.DataFrame({
    'Operacion': list(polars_times.keys()),
    'Polars_s': list(polars_times.values()),
    'Pandas_s': list(pandas_times.values())
})

df_results = df_results.with_columns(
    (pl.col('Pandas_s') / pl.col('Polars_s')).alias('Speedup_Polars')
)

print(df_results)